In [1]:
import os
import sys
import pickle
import argparse
import keras
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.stats import loguniform

sys.path.insert(1, 'scripts')
sys.path.insert(1, 'stella')

import stella

2025-01-22 12:47:23.897052: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/astro/phrdhx/micromamba/envs/nets2/lib/python3.8/site-packages/lightkurve/config/__init__.py:119: UserWarning: The default Lightkurve cache directory, used by download(), etc., has been moved to /home/astro/phrdhx/.lightkurve/cache. Please move all the files in the legacy directory /home/astro/phrdhx/.lightkurve-cache to the new location and remove the legacy directory. Refer to https://docs.lightkurve.org/reference/config.html#default-cache-directory-migration for more information.
  warnings.warn(


## Load dataset

In [2]:
with open("../datasets/comets-plus-extras-opt.pkl", "rb") as file:
    ds = pickle.load(file)

In [3]:
data = ds['dataset']

## Load optimised layers

In [4]:
import optuna

stored = 'sqlite:///../comets-plus-extras-opt.db'
study_name = '../comets-plus-extras-opt'
study = optuna.load_study(
    study_name=None,
    storage=stored
)

# Get the best hyperparameters
best_params = study.best_params

[I 2025-01-22 12:47:39,501] Study name was omitted but trying to load 'comets-plus-extras-opt.db' because that was the only study found in the storage.


In [5]:
best_params

{'dropout': 0.10417493570995731,
 'l2_lambda': 2.230316343500575e-05,
 'learning_rate': 0.00031221048872947035,
 'batch_size': 128}

## Load CNN

### Optimised layer

In [6]:
layers = [
    tf.keras.layers.Conv1D(
        filters=16,
        kernel_size=7,
        activation='relu',
        padding="same",
        input_shape=(168, 1),
        kernel_regularizer=tf.keras.regularizers.l2(2.230316343500575e-05)
    ),
    tf.keras.layers.MaxPooling1D(pool_size=2),
    tf.keras.layers.Dropout(0.10417493570995731),
    tf.keras.layers.Conv1D(
        filters=64,
        kernel_size=3,
        activation='relu',
        padding="same",
        kernel_regularizer=tf.keras.regularizers.l2(2.230316343500575e-05)
    ),
    tf.keras.layers.MaxPooling1D(pool_size=2),
    tf.keras.layers.Dropout(0.10417493570995731),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(32, 
                         activation='relu',
                         kernel_regularizer=tf.keras.regularizers.l2(2.230316343500575e-05)),
    tf.keras.layers.Dropout(0.10417493570995731),
    tf.keras.layers.Dense(1, activation="sigmoid")
]

In [7]:
cnn = stella.ConvNN(
    output_dir='../cnn-models/',
    ds=data,
    layers=layers  # Pass in your custom layers
)

In [8]:
val_predictions, histories = cnn.cross_validation(
    seed=49,
    epochs=200, 
    batch_size=512,
    n_splits=5, 
    shuffle=True,
    pred_test=False,
    save=True
)


Original label distribution:
Label 0: 26944 samples
Label 1: 45000 samples
Label 2: 4500 samples
Label 3: 4593 samples
Label 4: 4500 samples
Label 99: 4500 samples
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv1d (Conv1D)             (None, 168, 16)           128       
                                                                 
 max_pooling1d (MaxPooling1  (None, 84, 16)            0         
 D)                                                              
                                                                 
 dropout (Dropout)           (None, 84, 16)            0         
                                                                 
 conv1d_1 (Conv1D)           (None, 84, 64)            3136      
                                                                 
 max_pooling1d_1 (MaxPoolin  (None, 42, 64)            0         
 g1D)                  

/home/astro/phrdhx/micromamba/envs/nets2/lib/python3.8/site-packages/keras/src/engine/training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


563/563 [==============================] - 1s 2ms/step


FileNotFoundError: [Errno 2] No such file or directory: '/home/astro/phrdhx/nets2/notebooks/plots/confusion_matrix_2xN_seed49.png'